# Inspect HR

Samples the change feed and the Germany parental-leave policy, then compares
the source with `kbs-raw-snapshots`. The public records must not contain any
name in `hr_demo.FORBIDDEN`. Jupyter can reach floci on `kbs-aws`. It is not
on `kbs-data`.

Run this after the connector has caught up. The last cell waits
`hr_demo.CATCHUP_WAIT` seconds (three minutes). A 10,000 row load can take longer.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "hr_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import hr_demo


## Source


In [ ]:
sample = hr_demo.admin("GET", "/changes", params={"limit": 5})
assert sample["items"], "the change feed is empty"
found = hr_demo.nested_keys(sample) & hr_demo.FORBIDDEN
assert not found, found
item = sample["items"][0]
print("denied", sorted(hr_demo.FORBIDDEN))
print(item["object_type"], item["id"], sorted(item["record"]))

policy = hr_demo.admin("GET", "/policies/pol-parental-leave-de")
print(policy["title"], policy["region"], policy["version"])
content = hr_demo.request("GET", "/policies/pol-parental-leave-de/content")
assert "Elternzeit" in content.text
print(content.headers["content-type"], len(content.content), "bytes")


## Snapshots


In [ ]:
import json

KEY = "raw/hr/policy/pol-parental-leave-de.md"
hr_demo.wait_until(
    lambda: hr_demo.snapshot(KEY) is not None,
    hr_demo.WRITE_WAIT,
    "parental leave snapshot",
)
head = hr_demo.snapshot(KEY)
print("bucket", hr_demo.BUCKET)
print(KEY)
print(head["Metadata"])
assert b"Elternzeit" in hr_demo.object_body(KEY)

def caught_up() -> bool:
    return len(hr_demo.list_keys("raw/hr/")) == hr_demo.source_total()

hr_demo.wait_until(caught_up, hr_demo.CATCHUP_WAIT, "snapshot count matches the source")
keys = hr_demo.list_keys("raw/hr/")
source = hr_demo.source_total()
person_keys = [key for key in keys if "/person/" in key]
if person_keys:
    record = json.loads(hr_demo.object_body(person_keys[0]))
    leaked = set(record) & hr_demo.FORBIDDEN
    assert not leaked, leaked
drift = len(keys) - source
print("source", source, "snapshots", len(keys), "drift", drift)
assert drift == 0
